In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.integrate import trapezoid
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data/raw/hitting'
OUT = ROOT / 'results/hitting_phase_decomposition'
FIG = ROOT / 'figures/hitting_phase_decomposition'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)
AUTO_DOWNLOAD = True
OUTER_FOLDS = 5
INNER_FOLDS = 4
CV_SEEDS = (20261002, 20261003)
ALPHAS = (0.1, 1., 10., 100., 1000.)
BOOTSTRAPS = 2000
SEED = 20261002
SELECTED_SWING = None
PRE_CONTACT_LOOKBACK_S = .150
SPLIT_FRACTIONS = (.5, .4, .6)
FOLLOWTHROUGH_S = .050
PHASE_NAMES = ('approach', 'landing', 'early_postplant', 'late_postplant')
EVENTS = ['fp_10_time', 'fp_100_time', 'contact_time']
ANGLE_COLS = ['pelvis_angle_z', 'torso_angle_z', 'torso_angle_x', 'pelvis_angle_x',
              'lead_knee_angle_x', 'rear_elbow_angle_x']
VELO_COLS = ['pelvis_angular_velocity_z', 'torso_angular_velocity_z', 'lead_knee_angular_velocity_x', 'rear_elbow_angular_velocity_x']
OUTCOMES = {'contact_bat_speed': 'bat_speed_mph_contact_x',
            'blast_bat_speed': 'blast_bat_speed_mph_x', 'exit_velocity': 'exit_velo_mph_x'}
CONTROLS = ['height_m', 'mass_kg', 'bat_weight_oz', 'bat_length_in', 'left_handed']
ARCHIVES = {'joint_angles': '98a405c5c4a27c1a599d3e30b39a3087180cc4d4bb69faf62204a1c06881be95',
            'joint_velos': '3822dbce30988bc554ee533e5ccc12a5a3e92a53e185fdcb1cc8ad218dd464e4'}
print('Swing Phase Profiles & Bat Speed')
print('Official contact/plant events; fractional post-plant windows are analytical phases, not annotated swing initiation.')
print('Body motion predicts contact bat speed; Blast speed and exit velocity are separate secondary outcomes.')
print('First run downloads about 155 MB of missing archives; later runs reuse local CSVs.')

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def download(url, path):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {path}; set AUTO_DOWNLOAD=True or supply the official file.')
    print(f'Downloading {path.name} ...', flush=True)
    temporary = path.with_suffix(path.suffix + '.part')
    urlretrieve(url, temporary)
    temporary.replace(path)

base = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/'
csv_base = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_hitting/data/'
for name, digest in ARCHIVES.items():
    target = RAW / (name + '.csv')
    if not target.exists():
        archive = RAW / ('hitting_' + name + '.zip')
        if not archive.exists():
            download(base + archive.name, archive)
        if sha256(archive) != digest:
            raise ValueError(f'{archive.name}: Dataset v1 archive hash mismatch')
        with zipfile.ZipFile(archive) as z:
            matches = [n for n in z.namelist() if Path(n).name == target.name]
            if len(matches) != 1:
                raise ValueError(f'Expected exactly one {target.name}')
            temporary = target.with_suffix('.csv.part')
            with z.open(matches[0]) as source, temporary.open('wb') as dest:
                shutil.copyfileobj(source, dest)
            temporary.replace(target)
for name, remote in [('metadata.csv', 'metadata.csv'), ('poi_metrics.csv', 'poi/poi_metrics.csv')]:
    if not (RAW / name).exists():
        download(csv_base + remote, RAW / name)

metadata = pd.read_csv(RAW / 'metadata.csv', dtype={'session_swing': str, 'user': str, 'session': str})
poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_swing': str})
angles = pd.read_csv(RAW / 'joint_angles.csv', usecols=['session_swing', 'time'] + EVENTS + ANGLE_COLS,
                     dtype={'session_swing': str})
velos = pd.read_csv(RAW / 'joint_velos.csv', usecols=['session_swing', 'time'] + EVENTS + VELO_COLS,
                    dtype={'session_swing': str})
for name, table in [('metadata', metadata), ('POI', poi)]:
    if table.session_swing.duplicated().any():
        raise ValueError(f'Duplicate swing IDs in {name}')
for name, table in [('angles', angles), ('velos', velos)]:
    if table.duplicated(['session_swing', 'time']).any():
        raise ValueError(f'Duplicate swing/time pairs in {name}')
swings = metadata[['session_swing', 'user', 'session', 'hitter_side', 'session_mass_lbs',
                   'session_height_in', 'bat_weight_oz', 'bat_length_in']].merge(
    poi[['session_swing'] + list(OUTCOMES.values())], on='session_swing', how='left', validate='one_to_one')
swings['mass_kg'] = swings.session_mass_lbs * .45359237
swings['height_m'] = swings.session_height_in * .0254
swings['left_handed'] = swings.hitter_side.eq('L').astype(float)
angle_groups = {k: g.sort_values('time') for k, g in angles.groupby('session_swing', sort=False)}
velo_groups = {k: g.sort_values('time') for k, g in velos.groupby('session_swing', sort=False)}
print(f'Input: {len(swings)} swings from {swings.user.nunique()} hitters')
print('Signal outcome columns are not loaded as predictors.')

In [ ]:
def phase_window(g, columns, lo, hi):
    if g is None or not np.isfinite([lo, hi]).all() or lo >= hi:
        raise ValueError('missing signal or invalid phase boundaries')
    t = g.time.to_numpy(float)
    if not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('invalid time grid')
    if lo < t[0] or hi > t[-1]:
        raise ValueError('phase outside sampled recording')
    dt = np.median(np.diff(t))
    if not np.allclose(np.diff(t), dt, atol=0.00011, rtol=0.03):
        raise ValueError('irregular sampling')
    left = max(0, np.searchsorted(t, lo, side='right') - 1)
    right = min(len(t), np.searchsorted(t, hi, side='left') + 1)
    tt, values = t[left:right], g[columns].to_numpy(float)[left:right]
    if len(tt) < 2 or not np.isfinite(values).all():
        raise ValueError('missing phase samples or signal gaps')
    for i, col in enumerate(columns):
        if col in ('pelvis_angle_z', 'torso_angle_z'):
            values[:, i] = np.rad2deg(np.unwrap(np.deg2rad(values[:, i])))
    grid = np.r_[lo, tt[(tt > lo) & (tt < hi)], hi]
    sampled = np.column_stack([np.interp(grid, tt, values[:, i]) for i in range(len(columns))])
    return grid, sampled

def event_values(g):
    if g is None:
        raise ValueError('missing angle signal')
    values = {}
    for name in EVENTS:
        if g[name].nunique(dropna=False) != 1:
            raise ValueError('inconsistent event annotation')
        values[name] = float(g[name].iloc[0])
    return values


def phase_bounds(events, fraction=.5):
    ffc, plant, contact = [events[n] for n in EVENTS]
    if not np.isfinite([ffc, plant, contact]).all() or not ffc < plant < contact:
        raise ValueError('missing or unordered official events')
    if not 0 < fraction < 1:
        raise ValueError('post-plant split fraction must be between zero and one')
    split = plant + fraction * (contact - plant)
    return {'approach': (ffc - PRE_CONTACT_LOOKBACK_S, ffc), 'landing': (ffc, plant),
            'early_postplant': (plant, split), 'late_postplant': (split, contact)}

def extract_phase(ag, vg, lo, hi):
    ta, a = phase_window(ag, ANGLE_COLS, lo, hi)
    tv, v = phase_window(vg, VELO_COLS, lo, hi)
    if hi - lo < 2 / 360:
        raise ValueError('phase shorter than two marker sampling intervals')
    f = {'duration_ms': 1000 * (hi - lo)}
    for i, col in enumerate(ANGLE_COLS):
        f[col + '_change_deg'] = a[-1, i] - a[0, i]
    separation = a[:, 1] - a[:, 0]
    separation = np.rad2deg(np.unwrap(np.deg2rad(separation)))
    separation -= 360 * np.round(separation[0] / 360)
    f['torso_minus_pelvis_mean_deg'] = trapezoid(separation, ta) / (hi - lo)
    f['torso_minus_pelvis_change_deg'] = separation[-1] - separation[0]
    for i, col in enumerate(VELO_COLS):
        f[col + '_mean_dps'] = trapezoid(v[:, i], tv) / (hi - lo)
        direction = -1 if col in ('lead_knee_angular_velocity_x', 'rear_elbow_angular_velocity_x') else 1
        f[col + '_directional_peak_dps'] = (direction * v[:, i]).max()
    return f

def build_features(fraction):
    records, audit_rows = [], []
    for swing in swings.itertuples(index=False):
        key = swing.session_swing
        try:
            if pd.isna(swing.user) or swing.hitter_side not in ('L', 'R') or not np.isfinite(
                    [getattr(swing, c) for c in CONTROLS]).all() or min(
                    swing.mass_kg, swing.height_m, swing.bat_weight_oz, swing.bat_length_in) <= 0:
                raise ValueError('missing hitter identity or body/bat controls')
            ag, vg = angle_groups.get(key), velo_groups.get(key)
            events = event_values(ag)
            bounds = phase_bounds(events, fraction)
            other = event_values(vg)
            if any(not np.isclose(events[n], other[n], atol=.0001, rtol=0) for n in EVENTS):
                raise ValueError('event disagreement across signal tables')
            row = dict(plant_contact_outside_80_200ms=not .080 <= events['contact_time'] - events['fp_100_time'] <= .200, session_swing=key, hitter_id=str(swing.user), session_id=str(swing.session),
                       hitter_side=swing.hitter_side, split_fraction=fraction,
                       **{c: getattr(swing, c) for c in CONTROLS},
                       **{label: getattr(swing, col) for label, col in OUTCOMES.items()}, **events)
            for phase, (lo, hi) in bounds.items():
                row.update({phase + '__' + c: v for c, v in extract_phase(ag, vg, lo, hi).items()})
            records.append(row)
            status = 'retained'
        except (ValueError, TypeError, KeyError) as exc:
            status = str(exc)
        audit_rows.append(dict(session_swing=key, split_fraction=fraction, status=status))
    return pd.DataFrame(records), pd.DataFrame(audit_rows)

feature_sets, audits = {}, []
for fraction in SPLIT_FRACTIONS:
    table, audit = build_features(fraction)
    if table.empty:
        display(audit.status.value_counts())
        raise ValueError(f'No swings retained at split {fraction}')
    feature_sets[fraction] = table
    audits.append(audit)
    print(f'Split {fraction:.0%}: retained {len(table)} swings from {table.hitter_id.nunique()} hitters')
features = feature_sets[.5]
extraction_audit = pd.concat(audits, ignore_index=True)
extraction_audit.to_csv(OUT / 'extraction_audit.csv', index=False)
pd.concat(feature_sets.values(), ignore_index=True).to_csv(OUT / 'swing_phase_features.csv', index=False)
display(extraction_audit[extraction_audit.split_fraction.eq(.5)].status.value_counts().rename('swings').to_frame())
followthrough_rows = []
for row in features.itertuples(index=False):
    try:
        f = extract_phase(angle_groups[row.session_swing], velo_groups[row.session_swing],
                          row.contact_time, row.contact_time + FOLLOWTHROUGH_S)
        followthrough_rows.append(dict(session_swing=row.session_swing, hitter_id=row.hitter_id, status='retained', **f))
    except ValueError as exc:
        followthrough_rows.append(dict(session_swing=row.session_swing, hitter_id=row.hitter_id, status=str(exc)))
pd.DataFrame(followthrough_rows).to_csv(OUT / 'followthrough_descriptive.csv', index=False)
print(f'Duration sensitivity flags: {features.plant_contact_outside_80_200ms.sum()} swings outside 80–200 ms plant-to-contact')
print('Approach is a fixed 150 ms window before front-foot contact, not a measured stride onset.')
print('Pelvis/torso forward rotation is positive; knee/elbow extension is negative in OBP hitting conventions.')
print('No head markers are present in these full-signal inputs; head stability remains in study 07.')

In [ ]:
def hitter_weights(ids):
    ids = pd.Series(np.asarray(ids))
    return (1 / ids.map(ids.value_counts())).to_numpy(float)

def group_splits(ids, folds, seed):
    ids = np.asarray(ids)
    unique = np.unique(ids)
    if len(unique) < folds:
        raise ValueError('Too few hitters for grouped validation')
    shuffled = np.random.default_rng(seed).permutation(unique)
    for held_out in np.array_split(shuffled, folds):
        test = np.flatnonzero(np.isin(ids, held_out))
        train = np.flatnonzero(~np.isin(ids, held_out))
        if set(ids[train]) & set(ids[test]):
            raise AssertionError('Hitter leakage')
        yield train, test

def group_mae(y, prediction, ids):
    return pd.DataFrame({'id': ids, 'error': np.abs(np.asarray(y) - prediction)}).groupby('id').error.mean().mean()

def fit_ridge(x, y, ids, alpha):
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(x, y, ridge__sample_weight=hitter_weights(ids))
    return model

def tune_alpha(x, y, ids, seed):
    splits = list(group_splits(ids, INNER_FOLDS, seed))
    scores = []
    for alpha in ALPHAS:
        predicted = np.full(len(y), np.nan)
        for train, test in splits:
            model = fit_ridge(x.iloc[train], y[train], ids[train], alpha)
            predicted[test] = model.predict(x.iloc[test])
        scores.append(group_mae(y, predicted, ids))
    return ALPHAS[int(np.argmin(scores))]


phase_columns = {p: [c for c in features if c.startswith(p + '__')] for p in PHASE_NAMES}
all_phase_columns = sum(phase_columns.values(), [])
if any('bat_speed' in c or 'followthrough' in c or 'exit_velocity' in c for c in all_phase_columns):
    raise AssertionError('Outcome or post-contact predictor leakage')
specs = {'training_mean': [], 'body_bat_controls': CONTROLS,
         **{p + '_only': CONTROLS + phase_columns[p] for p in PHASE_NAMES},
         'all_phases': CONTROLS + all_phase_columns,
         **{'omit_' + p: CONTROLS + sum([phase_columns[q] for q in PHASE_NAMES if q != p], [])
            for p in PHASE_NAMES}}
common_ids = set.intersection(*(set(t.session_swing) for t in feature_sets.values()))
analysis_sets = [('main', f, t) for f, t in feature_sets.items()] + [
    ('duration_sensitivity', .5, features[~features.plant_contact_outside_80_200ms])]
prediction_rows, tuning_rows, fold_rows, cohort_rows = [], [], [], []
for outcome in OUTCOMES:
    eligible = features[np.isfinite(features[outcome]) & (features[outcome] > 0) &
                        features.session_swing.isin(common_ids)]
    ids_for_outcome = set(eligible.session_swing)
    if eligible.hitter_id.nunique() < 20:
        print(f'{outcome}: too few complete hitters; skipped')
        continue
    for analysis, fraction, table in analysis_sets:
        table = table[table.session_swing.isin(ids_for_outcome)].sort_values('session_swing').reset_index(drop=True)
        cohort_rows.append(dict(outcome=outcome, analysis=analysis, split_fraction=fraction,
                                n_swings=len(table), n_hitters=table.hitter_id.nunique()))
        x, y, ids = table, table[outcome].to_numpy(float), table.hitter_id.to_numpy()
        for repeat, seed in enumerate(CV_SEEDS):
            for fold, (train, test) in enumerate(group_splits(ids, OUTER_FOLDS, seed)):
                fold_rows.append(dict(outcome=outcome, analysis=analysis, split_fraction=fraction, repeat=repeat, fold=fold,
                                      training_hitters='|'.join(sorted(set(ids[train]))),
                                      test_hitters='|'.join(sorted(set(ids[test])))))
                for label, cols in specs.items():
                    if cols:
                        alpha = tune_alpha(x[cols].iloc[train], y[train], ids[train], seed + fold)
                        model = fit_ridge(x[cols].iloc[train], y[train], ids[train], alpha)
                        pred = model.predict(x[cols].iloc[test])
                    else:
                        alpha = np.nan
                        pred = np.repeat(np.average(y[train], weights=hitter_weights(ids[train])), len(test))
                    tuning_rows.append(dict(outcome=outcome, analysis=analysis, split_fraction=fraction, repeat=repeat,
                                            fold=fold, model=label, alpha=alpha))
                    for j, index in enumerate(test):
                        prediction_rows.append(dict(outcome=outcome, analysis=analysis, split_fraction=fraction, repeat=repeat,
                            fold=fold, model=label, session_swing=table.session_swing.iloc[index], hitter_id=ids[index],
                            actual_mph=y[index], predicted_mph=pred[j]))
            print(f'{outcome}, {analysis}, split {fraction:.0%}: repeat {repeat + 1}/{len(CV_SEEDS)} complete', flush=True)
predictions = pd.DataFrame(prediction_rows)
if predictions.empty:
    raise ValueError('No validation predictions; inspect the cohort audit')
cohort_audit = pd.DataFrame(cohort_rows)
display(cohort_audit)
cohort_audit.to_csv(OUT / 'outcome_cohort_audit.csv', index=False)
predictions.to_csv(OUT / 'held_out_predictions.csv', index=False)
pd.DataFrame(tuning_rows).to_csv(OUT / 'ridge_tuning.csv', index=False)
pd.DataFrame(fold_rows).to_csv(OUT / 'validation_fold_audit.csv', index=False)

In [ ]:
def interval(values):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(SEED)
    draws = values[rng.integers(0, len(values), size=(BOOTSTRAPS, len(values)))].mean(axis=1)
    return values.mean(), *np.quantile(draws, [.025, .975])


predictions['absolute_error'] = abs(predictions.actual_mph - predictions.predicted_mph)
metric_rows, comparison_rows = [], []
for (outcome, analysis, fraction, label), g in predictions.groupby(['outcome', 'analysis', 'split_fraction', 'model']):
    per_hitter = g.groupby('hitter_id').absolute_error.mean()
    mean, low, high = interval(per_hitter)
    weights = hitter_weights(g.hitter_id)
    metric_rows.append(dict(outcome=outcome, analysis=analysis, split_fraction=fraction, model=label,
        hitter_weighted_mae_mph=mean, bootstrap_ci_low=low, bootstrap_ci_high=high,
        hitter_weighted_rmse_mph=np.sqrt(np.average((g.actual_mph - g.predicted_mph) ** 2, weights=weights)),
        n_hitters=g.hitter_id.nunique(), n_swings=g.session_swing.nunique()))
for (outcome, analysis, fraction), g in predictions.groupby(['outcome', 'analysis', 'split_fraction']):
    errors = g.groupby(['hitter_id', 'model']).absolute_error.mean().unstack('model')
    for phase in PHASE_NAMES:
        delta = errors['omit_' + phase] - errors['all_phases']
        estimate, lo, hi = interval(delta.dropna())
        comparison_rows.append(dict(outcome=outcome, analysis=analysis, split_fraction=fraction, phase=phase,
            mae_increase_when_removed_mph=estimate, bootstrap_ci_low=lo, bootstrap_ci_high=hi))
    estimate, lo, hi = interval(errors.body_bat_controls - errors.all_phases)
    comparison_rows.append(dict(outcome=outcome, analysis=analysis, split_fraction=fraction, phase='all_vs_controls',
        mae_increase_when_removed_mph=estimate, bootstrap_ci_low=lo, bootstrap_ci_high=hi))
metrics = pd.DataFrame(metric_rows)
comparisons = pd.DataFrame(comparison_rows)
metrics.to_csv(OUT / 'model_metrics.csv', index=False)
comparisons.to_csv(OUT / 'phase_ablation.csv', index=False)
repeat_scores = predictions.groupby(['outcome', 'analysis', 'split_fraction', 'repeat', 'model', 'hitter_id']).absolute_error.mean().groupby(
    ['outcome', 'analysis', 'split_fraction', 'repeat', 'model']).mean().rename('mae_mph').reset_index()
repeat_scores.to_csv(OUT / 'repeat_model_metrics.csv', index=False)
print('Primary contact-bat-speed model comparisons at the 50% split:')
display(metrics[metrics.outcome.eq('contact_bat_speed') & metrics.split_fraction.eq(.5) & metrics.analysis.eq('main')].round(3))
print('Phase ablations across split definitions (positive means removing the phase worsens prediction):')
display(comparisons.round(3))
print('Intervals resample hitters from fixed held-out errors, after averaging repeats; they do not refit nested CV.')
print('Adjacent phases share information. Ablations are not causal percentages or shares of bat speed.')
print('Secondary outcomes use their own complete-outcome cohorts; compare models within each outcome.')

In [ ]:
selected = str(SELECTED_SWING) if SELECTED_SWING is not None else features.session_swing.iloc[0]
if selected not in set(features.session_swing):
    raise KeyError(f'Swing {selected} was not retained')
row = features.set_index('session_swing').loc[selected]
g = velo_groups[selected]
bounds = phase_bounds(row.to_dict(), .5)
colors = dict(zip(PHASE_NAMES, ['#609d78', '#e4ae48', '#467ca8', '#be5946']))
fig, ax = plt.subplots(figsize=(11, 5))
for phase, (lo, hi) in bounds.items():
    ax.axvspan(1000 * (lo - row.fp_100_time), 1000 * (hi - row.fp_100_time), alpha=.14,
               color=colors[phase], label=phase.replace('_', ' ').title())
for col, label in [('pelvis_angular_velocity_z', 'Pelvis'), ('torso_angular_velocity_z', 'Torso')]:
    mask = g.time.between(bounds['approach'][0], row.contact_time)
    ax.plot(1000 * (g.loc[mask, 'time'] - row.fp_100_time), g.loc[mask, col], label=label)
for time in [row.fp_10_time, row.fp_100_time, bounds['late_postplant'][0], row.contact_time]:
    ax.axvline(1000 * (time - row.fp_100_time), color='gray', linestyle=':', linewidth=1)
ax.set(xlabel='Time from foot plant (ms)', ylabel='Forward axial angular velocity (deg/s)',
       title=f'Swing {selected}: contact bat speed {row.contact_bat_speed:.1f} mph; analytical 50% split')
ax.legend(ncol=3, fontsize=9)
fig.tight_layout(); fig.savefig(FIG / 'phase_timeline.png', dpi=180); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
main = metrics[metrics.outcome.eq('contact_bat_speed') & metrics.split_fraction.eq(.5) & metrics.analysis.eq('main')].sort_values('hitter_weighted_mae_mph')
axes[0].barh(main.model.str.replace('_', ' '), main.hitter_weighted_mae_mph, color='#467ca8')
axes[0].set(xlabel='Held-out MAE (mph; lower is better)', title='Contact bat speed: models on the same swings')
main_contrast = comparisons[comparisons.outcome.eq('contact_bat_speed') & comparisons.split_fraction.eq(.5) & comparisons.analysis.eq('main') &
                            comparisons.phase.isin(PHASE_NAMES)]
delta = main_contrast.mae_increase_when_removed_mph.to_numpy()
axes[1].errorbar(delta, np.arange(len(main_contrast)),
    xerr=np.maximum(0, np.vstack([delta - main_contrast.bootstrap_ci_low, main_contrast.bootstrap_ci_high - delta])),
    fmt='o', color='#be5946')
axes[1].set_yticks(np.arange(len(main_contrast)), main_contrast.phase.str.replace('_', ' '))
axes[1].axvline(0, color='gray', linestyle=':')
axes[1].set(xlabel='MAE increase when phase is removed (mph)', title='Additional phase information; conditional 95% intervals')
fig.tight_layout(); fig.savefig(FIG / 'phase_model_comparison.png', dpi=180); plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sens = comparisons[comparisons.outcome.eq('contact_bat_speed') & comparisons.analysis.eq('main') & comparisons.phase.isin(PHASE_NAMES)]
for phase in PHASE_NAMES:
    sample = sens[sens.phase.eq(phase)].sort_values('split_fraction')
    ax.plot(sample.split_fraction * 100, sample.mae_increase_when_removed_mph, marker='o',
            label=phase.replace('_', ' ').title(), color=colors[phase])
ax.axhline(0, color='gray', linestyle=':')
ax.set(xlabel='Analytical boundary (% of plant-to-contact time)', ylabel='Held-out MAE increase on phase removal (mph)',
       title='Phase-definition sensitivity; matched swings and hitter folds')
ax.set_xticks([40, 50, 60]); ax.legend(ncol=2)
fig.tight_layout(); fig.savefig(FIG / 'phase_split_sensitivity.png', dpi=180); plt.show()

profiles = features.groupby('hitter_id')[CONTROLS + list(OUTCOMES) + all_phase_columns].mean()
profiles.to_csv(OUT / 'hitter_profiles.csv')
profiles[all_phase_columns].rank(pct=True).mul(100).to_csv(OUT / 'sample_relative_profile_percentiles.csv')
manifest = {'input_sha256': {name: sha256(RAW / name) for name in
    ['metadata.csv', 'poi_metrics.csv', 'joint_angles.csv', 'joint_velos.csv']},
    'archive_sha256_expected': ARCHIVES, 'dataset_tag': 'dataset-v1', 'outer_folds': OUTER_FOLDS,
    'inner_folds': INNER_FOLDS, 'cv_seeds': CV_SEEDS, 'ridge_alphas': ALPHAS, 'bootstraps': BOOTSTRAPS,
    'bootstrap_seed': SEED, 'split_fractions': SPLIT_FRACTIONS, 'pre_foot_contact_lookback_s': PRE_CONTACT_LOOKBACK_S,
    'followthrough_s': FOLLOWTHROUGH_S, 'duration_sensitivity_plant_contact_ms': [80, 200], 'predictor_columns': specs,
    'outcomes': OUTCOMES, 'cohorts': cohort_audit.to_dict('records'),
    'versions': {name: __import__(name).__version__ for name in ['numpy', 'pandas', 'scipy', 'sklearn']}}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
print(f'Tables and profiles: {OUT}')
print(f'Figures: {FIG}')
print('Sample-relative percentiles are descriptive, not ideal mechanics or validated coaching targets.')